Raw Data Profile

Quick inspection notebook for `data/raw/books_goodreads.csv`.

Checks covered:
- Column names
- Missing values
- Duplicate books
- Weird date formats
- Text quality in descriptions and genres


In [2]:
import ast
import re
from pathlib import Path

import pandas as pd

DATA_PATH = Path('../data/raw/books_goodreads.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/raw/books_goodreads.csv')

# Keep everything as strings for the raw-data profile. Empty strings stay empty
# so missingness checks match the scraped CSV instead of pandas' default NA rules.
df = pd.read_csv(DATA_PATH, dtype='string', keep_default_na=False)
columns = df.columns.tolist()

df.shape


(52478, 25)

## Column Names

In [3]:
columns


['bookId',
 'title',
 'series',
 'author',
 'rating',
 'description',
 'language',
 'isbn',
 'genres',
 'characters',
 'bookFormat',
 'edition',
 'pages',
 'publisher',
 'publishDate',
 'firstPublishDate',
 'awards',
 'numRatings',
 'ratingsByStars',
 'likedPercent',
 'setting',
 'coverImg',
 'bbeScore',
 'bbeVotes',
 'price']

## Missing Values

The raw CSV uses empty strings for many missing values, so this counts blank strings after stripping whitespace.

In [5]:
blank_mask = df.apply(lambda col: col.str.strip().eq(''))

missing_summary = (
    pd.DataFrame({
        'blank_rows': blank_mask.sum(),
        'blank_percent': (blank_mask.mean() * 100).round(1),
    })
    .sort_values('blank_rows', ascending=False)
)

missing_summary


,blank_rows,blank_percent
edition,47523,90.6
series,29009,55.3
firstPublishDate,21326,40.6
price,14365,27.4
language,3806,7.3
publisher,3695,7.0
pages,2347,4.5
bookFormat,1473,2.8
description,1337,2.5
publishDate,880,1.7


## Duplicate Books

These checks are intentionally conservative. `title` duplicates can be legitimate editions/translations, while duplicate `bookId` rows are stronger evidence of repeated records.

In [6]:
def normalize_text(series):
    return series.fillna('').str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def duplicate_summary(key_columns, ignore_blank=True):
    key_columns = list(key_columns)
    keys = df[key_columns].apply(normalize_text)
    if ignore_blank:
        keys = keys[keys.ne('').all(axis=1)]

    group_sizes = keys.value_counts(dropna=False)
    duplicate_groups = group_sizes[group_sizes > 1]
    return pd.Series({
        'key_columns': ', '.join(key_columns),
        'duplicate_rows': int(duplicate_groups.sum()),
        'duplicate_groups': int(duplicate_groups.shape[0]),
    })

pd.DataFrame([
    duplicate_summary(('bookId',)),
    duplicate_summary(('title', 'author')),
    duplicate_summary(('isbn',)),
    duplicate_summary(('title',)),
])


,key_columns,duplicate_rows,duplicate_groups
0,bookId,108,54
1,"title, author",183,89
2,isbn,4460,54
3,title,4316,1710


In [8]:
placeholder_isbn = '9999999999999'
isbn = normalize_text(df['isbn'])
isbn_counts = isbn[isbn.ne('')].value_counts()

pd.DataFrame({
    'metric': ['placeholder_isbn_rows', 'duplicate_non_placeholder_isbn_groups'],
    'value': [
        int(isbn_counts.get(placeholder_isbn, 0)),
        int(isbn_counts[(isbn_counts.index != placeholder_isbn) & (isbn_counts > 1)].shape[0]),
    ],
})


,metric,value
0,placeholder_isbn_rows,4354
1,duplicate_non_placeholder_isbn_groups,53


## Date Formats

The date columns mix several formats. The parser below is for profiling only; final cleaning should preserve raw values and apply explicit century rules for two-digit years.

In [9]:
def date_pattern(value):
    value = str(value or '').strip()
    if not value:
        return 'blank'
    if re.fullmatch(r'\d{1,2}/\d{1,2}/\d{2}', value):
        return 'M/D/YY'
    if re.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}', value):
        return 'M/D/YYYY'
    if re.fullmatch(r'\d{4}', value):
        return 'YYYY'
    if re.fullmatch(r'[A-Za-z]+ \d{1,2}(st|nd|rd|th)?,? \d{4}', value):
        return 'Month D YYYY'
    if re.fullmatch(r'[A-Za-z]+ \d{4}', value):
        return 'Month YYYY'
    return 'other'

def date_profile(column):
    values = df[column].fillna('').str.strip()
    normalized = values.str.replace(r'(\d)(st|nd|rd|th)', r'\1', regex=True)
    parsed = pd.to_datetime(normalized.mask(normalized.eq('')), errors='coerce', format='mixed')
    nonblank = values.ne('')
    failures = values[nonblank & parsed.isna()]
    years = parsed.dropna().dt.year
    future_like = df.loc[parsed.dt.year.gt(2026).fillna(False), ['title', column]].copy()
    future_like['parsed_year'] = parsed[parsed.dt.year.gt(2026).fillna(False)].dt.year.astype('int64').values

    return {
        'patterns': values.map(date_pattern).value_counts(),
        'nonblank': int(nonblank.sum()),
        'parseable_nonblank': int((nonblank & parsed.notna()).sum()),
        'sample_failures': failures.head(20).tolist(),
        'year_range': (int(years.min()), int(years.max())) if not years.empty else None,
        'sample_future_like_after_naive_parse': future_like.head(20),
    }

date_profiles = {column: date_profile(column) for column in ['publishDate', 'firstPublishDate']}

pd.DataFrame([
    {
        'column': column,
        'nonblank': profile['nonblank'],
        'parseable_nonblank': profile['parseable_nonblank'],
        'year_range': profile['year_range'],
        'sample_failures': profile['sample_failures'][:5],
    }
    for column, profile in date_profiles.items()
])


,column,nonblank,parseable_nonblank,year_range,sample_failures
0,publishDate,51598,50693,"(1820, 2075)","[Published, Published, Published, Best Books t..."
1,firstPublishDate,31152,31135,"(1704, 2075)","[14th 1857, 1181, 1230, 1614, 61]"


### Date Pattern Counts

In [11]:
pd.DataFrame({
    column: profile['patterns']
    for column, profile in date_profiles.items()
}).fillna(0).astype('int64')


,publishDate,firstPublishDate
M/D/YY,818,28780
Month D YYYY,43314,1186
Month YYYY,2199,151
YYYY,4362,1031
blank,880,21326
other,905,4


### Future-Looking Dates After Naive Parsing

In [12]:
date_profiles['firstPublishDate']['sample_future_like_after_naive_parse']


,title,firstPublishDate,parsed_year
2,To Kill a Mockingbird,07/11/60,2060
6,Animal Farm,08/17/45,2045
7,The Chronicles of Narnia,10/28/56,2056
8,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,10/20/55,2055
9,Gone with the Wind,06/30/36,2036
12,The Giving Tree,10/28/64,2064
13,Wuthering Heights,12/28/47,2047
17,Alice's Adventures in Wonderland & Through the...,10/28/71,2071
18,Jane Eyre,10/16/47,2047
19,Les Misérables,10/28/62,2062


## Description Text Quality

In [13]:
def text_profile(column):
    values = df[column].fillna('').str.strip()
    nonblank = values[values.ne('')]
    lengths = nonblank.str.len()

    return pd.Series({
        'blank': int(values.eq('').sum()),
        'short_nonblank_20_chars_or_less': int(lengths.le(20).sum()),
        'literal_null_strings': int(nonblank.str.lower().isin(['nan', 'none', 'null']).sum()),
        'html_or_entity_markers': int(nonblank.str.contains(r'<br|&amp;|&quot;', case=False, regex=True).sum()),
        'missing_space_after_period_pattern': int(nonblank.str.contains(r'[a-z]\.[A-Z]', regex=True).sum()),
        'length_min': int(lengths.min()) if not lengths.empty else None,
        'length_median': int(lengths.median()) if not lengths.empty else None,
        'length_max': int(lengths.max()) if not lengths.empty else None,
    })

text_profile('description')


blank                                  1337
short_nonblank_20_chars_or_less          98
literal_null_strings                      0
html_or_entity_markers                    0
missing_space_after_period_pattern    21284
length_min                                2
length_median                           791
length_max                            24733
dtype: int64

## Genre Quality

In [14]:
def parse_list(value):
    value = str(value or '').strip()
    if not value:
        return []
    parsed = ast.literal_eval(value)
    if not isinstance(parsed, list):
        raise ValueError('Expected list-like value')
    return parsed

parsed_genres = []
malformed_genres = []

for value in df['genres']:
    try:
        parsed_genres.append(parse_list(value))
    except Exception:
        malformed_genres.append(value)
        parsed_genres.append([])

genre_lengths = pd.Series([len(genres) for genres in parsed_genres], name='genre_count')
genres_exploded = pd.Series(parsed_genres).explode().dropna()
genres_exploded = genres_exploded[genres_exploded.ne('')]

genre_summary = pd.Series({
    'malformed_genre_rows': len(malformed_genres),
    'empty_genre_lists': int(genre_lengths.eq(0).sum()),
    'genre_count_min': int(genre_lengths.min()),
    'genre_count_median': int(genre_lengths.median()),
    'genre_count_max': int(genre_lengths.max()),
})

genre_summary


malformed_genre_rows       0
empty_genre_lists       4623
genre_count_min            0
genre_count_median        10
genre_count_max           10
dtype: int64

### Top Genres

In [15]:
genres_exploded.value_counts().head(30)


Fiction                    31638
Romance                    15495
Fantasy                    15046
Young Adult                11869
Contemporary               10520
Nonfiction                  8251
Adult                       8246
Novels                      7805
Mystery                     7702
Historical Fiction          7665
Audiobook                   7307
Classics                    6902
Adventure                   6452
Historical                  6383
Paranormal                  6030
Literature                  5836
Science Fiction             5374
Childrens                   5226
Thriller                    4587
Magic                       4248
Humor                       4227
History                     3685
Crime                       3675
Contemporary Romance        3624
Suspense                    3474
Urban Fantasy               3458
Middle Grade                3389
Chick Lit                   3358
Science Fiction Fantasy     3302
Supernatural                3196
Name: coun

## Numeric Oddities

In [16]:
def numeric_profile(column):
    raw = df[column].fillna('').str.strip()
    nonblank = raw.ne('')
    numeric = pd.to_numeric(raw.str.replace(',', '', regex=False), errors='coerce')
    bad = raw[nonblank & numeric.isna()]

    return pd.Series({
        'blank': int(raw.eq('').sum()),
        'nonblank_numeric': int((nonblank & numeric.notna()).sum()),
        'bad_nonblank': int(bad.shape[0]),
        'min': numeric.min(),
        'max': numeric.max(),
        'sample_bad_values': bad.head(5).tolist(),
    })

pd.DataFrame({
    column: numeric_profile(column)
    for column in ['rating', 'pages', 'numRatings', 'likedPercent', 'bbeScore', 'bbeVotes', 'price']
}).T


,blank,nonblank_numeric,bad_nonblank,min,max,sample_bad_values
rating,0,52478,0,0.0,5.0,[]
pages,2347,50108,23,0.0,14777.0,"[1 page, 1 page, 1 page, 1 page, 1 page]"
numRatings,0,52478,0,0,7048471,[]
likedPercent,622,51856,0,0.0,100.0,[]
bbeScore,0,52478,0,0,2993816,[]
bbeVotes,0,52478,0,-4,30516,[]
price,14365,38101,12,0.84,898.64,"[1.189.88, 1.743.28, 2.179.10, 1.743.28, 1.307..."


## Data Quality: Key Fields

This audit focuses on the README fields that matter most for the project question: publication era, youth-fiction theme signals, and popularity/cultural-memory scoring.

Note: In this notebook, `averageRating` is mapped to the raw `rating` column.


In [17]:
key_field_map = {
    'title': 'title',
    'author': 'author',
    'firstPublishDate': 'firstPublishDate',
    'genres': 'genres',
    'description': 'description',
    'numRatings': 'numRatings',
    'averageRating': 'averageRating' if 'averageRating' in df.columns else 'rating',
    'bbeVotes': 'bbeVotes',
    'bbeScore': 'bbeScore',
}

key_field_map


{'title': 'title',
 'author': 'author',
 'firstPublishDate': 'firstPublishDate',
 'genres': 'genres',
 'description': 'description',
 'numRatings': 'numRatings',
 'averageRating': 'rating',
 'bbeVotes': 'bbeVotes',
 'bbeScore': 'bbeScore'}

### Field-Level Audit

Compact audit for completeness, parseability, suspicious values, and immediate modeling risk.


In [18]:
def blank_count(column):
    return int(df[column].fillna('').str.strip().eq('').sum())

def blank_percent(column):
    return round(blank_count(column) / len(df) * 100, 1)

def numeric_audit(column, expected_min=None, expected_max=None):
    raw = df[column].fillna('').str.strip()
    nonblank = raw.ne('')
    numeric = pd.to_numeric(raw.str.replace(',', '', regex=False), errors='coerce')
    invalid = nonblank & numeric.isna()
    out_of_range = pd.Series(False, index=df.index)
    if expected_min is not None:
        out_of_range = out_of_range | numeric.lt(expected_min).fillna(False)
    if expected_max is not None:
        out_of_range = out_of_range | numeric.gt(expected_max).fillna(False)

    return {
        'blank_rows': int(raw.eq('').sum()),
        'blank_percent': round(raw.eq('').mean() * 100, 1),
        'invalid_numeric_rows': int(invalid.sum()),
        'out_of_expected_range_rows': int(out_of_range.sum()),
        'min': numeric.min(),
        'median': numeric.median(),
        'max': numeric.max(),
        'zero_rows': int(numeric.eq(0).sum()),
    }

def parsed_list_audit(column):
    parsed_values = []
    malformed = 0
    for value in df[column]:
        try:
            parsed = parse_list(value)
        except Exception:
            malformed += 1
            parsed = []
        parsed_values.append(parsed)

    lengths = pd.Series([len(value) for value in parsed_values])
    return {
        'blank_rows': blank_count(column),
        'blank_percent': blank_percent(column),
        'malformed_list_rows': malformed,
        'empty_list_rows': int(lengths.eq(0).sum()),
        'median_items': int(lengths.median()),
        'max_items': int(lengths.max()),
    }

def date_audit(column):
    values = df[column].fillna('').str.strip()
    normalized = values.str.replace(r'(\d)(st|nd|rd|th)', r'\1', regex=True)
    parsed = pd.to_datetime(normalized.mask(normalized.eq('')), errors='coerce', format='mixed')
    nonblank = values.ne('')
    years = parsed.dropna().dt.year
    return {
        'blank_rows': int(values.eq('').sum()),
        'blank_percent': round(values.eq('').mean() * 100, 1),
        'parseable_nonblank_percent': round((nonblank & parsed.notna()).sum() / max(nonblank.sum(), 1) * 100, 1),
        'future_like_rows_after_naive_parse': int(parsed.dt.year.gt(2026).fillna(False).sum()),
        'year_min': int(years.min()) if not years.empty else None,
        'year_max': int(years.max()) if not years.empty else None,
        'top_patterns': values.map(date_pattern).value_counts().head(5).to_dict(),
    }

def description_audit(column):
    values = df[column].fillna('').str.strip()
    nonblank = values[values.ne('')]
    lengths = nonblank.str.len()
    return {
        'blank_rows': int(values.eq('').sum()),
        'blank_percent': round(values.eq('').mean() * 100, 1),
        'short_nonblank_20_chars_or_less': int(lengths.le(20).sum()),
        'missing_space_after_period_rows': int(nonblank.str.contains(r'[a-z]\.[A-Z]', regex=True).sum()),
        'median_length': int(lengths.median()) if not lengths.empty else None,
        'max_length': int(lengths.max()) if not lengths.empty else None,
    }

field_audit = {
    'title': {
        'blank_rows': blank_count('title'),
        'blank_percent': blank_percent('title'),
        'duplicate_title_rows': int(normalize_text(df['title']).duplicated(keep=False).sum()),
        'median_length': int(df['title'].str.strip().str.len().median()),
    },
    'author': {
        'blank_rows': blank_count('author'),
        'blank_percent': blank_percent('author'),
        'multi_credit_rows': int(df['author'].str.contains(',', regex=False).sum()),
        'role_marker_rows': int(df['author'].str.contains(r'\(.*?\)', regex=True).sum()),
    },
    'firstPublishDate': date_audit('firstPublishDate'),
    'genres': parsed_list_audit('genres'),
    'description': description_audit('description'),
    'numRatings': numeric_audit('numRatings', expected_min=0),
    'averageRating': numeric_audit(key_field_map['averageRating'], expected_min=0, expected_max=5),
    'bbeVotes': numeric_audit('bbeVotes', expected_min=0),
    'bbeScore': numeric_audit('bbeScore', expected_min=0),
}

field_audit_df = (
    pd.DataFrame.from_dict(field_audit, orient='index')
    .rename_axis('field')
    .reset_index()
)

field_audit_df


,field,blank_rows,blank_percent,duplicate_title_rows,median_length,multi_credit_rows,role_marker_rows,parseable_nonblank_percent,future_like_rows_after_naive_parse,year_min,...,max_items,short_nonblank_20_chars_or_less,missing_space_after_period_rows,max_length,invalid_numeric_rows,out_of_expected_range_rows,min,median,max,zero_rows
0,title,0,0.0,4316.0,18.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,author,0,0.0,NaN,NaN,10848.0,29056.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,firstPublishDate,21326,40.6,NaN,NaN,NaN,NaN,99.9,5001.0,1704.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,genres,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,description,1337,2.5,NaN,791.0,NaN,NaN,NaN,NaN,NaN,...,NaN,98.0,21284.0,24733.0,NaN,NaN,NaN,NaN,NaN,NaN
5,numRatings,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,2307.00,7048471.0,71.0
6,averageRating,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,4.03,5.0,71.0
7,bbeVotes,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,52.0,-4.0,1.00,30516.0,2.0
8,bbeScore,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,97.00,2993816.0,8.0


### Audit Notes

- `title` and `author` are complete, but should be normalized for duplicate detection and author-level grouping.
- `firstPublishDate` is central to era analysis but has many blanks and ambiguous two-digit years, so it needs explicit date cleaning before use.
- `genres` is structurally parseable, but empty genre lists and broad Goodreads shelves mean it should feed theme tags cautiously.
- `description` is useful for keyword/theme extraction after text cleanup; scraped paragraph joins create many missing-space patterns.
- `numRatings`, `averageRating`, and `bbeVotes` match the README popularity-score ingredients, but they represent Goodreads engagement rather than neutral market popularity.
- `bbeScore` is useful mainly as a Goodreads ranking/canonization signal; it is less interpretable than the simpler vote and rating fields.


## Feature Decision Table

Trust labels use four options: `keep as-is`, `clean heavily`, `use cautiously`, and `ignore`. These decisions are for the analysis described in the README, especially era grouping, theme detection, and popularity scoring.


In [ ]:
feature_decisions = pd.DataFrame([
    {
        'field': 'title',
        'raw_column': 'title',
        'decision': 'clean heavily',
        'why': 'Complete and valuable, but needs whitespace/case normalization and duplicate-aware canonical keys.',
        'planned_use': 'Display label, duplicate checks, keyword/title signals after normalization.',
    },
    {
        'field': 'author',
        'raw_column': 'author',
        'decision': 'clean heavily',
        'why': 'Complete, but many rows contain multiple contributors or role markers such as illustrator/introduction.',
        'planned_use': 'Extract primary author and optionally preserve full contributor string.',
    },
    {
        'field': 'firstPublishDate',
        'raw_column': 'firstPublishDate',
        'decision': 'clean heavily',
        'why': 'Essential for era grouping, but has many blanks, mixed formats, and ambiguous two-digit years.',
        'planned_use': 'Create cleaned first_publish_year; filter to 1997-2024 after validation.',
    },
    {
        'field': 'genres',
        'raw_column': 'genres',
        'decision': 'use cautiously',
        'why': 'Parseable list-like strings, but Goodreads shelves are broad, user-generated, and sometimes empty.',
        'planned_use': 'Input to custom theme buckets, combined with title and description keywords.',
    },
    {
        'field': 'description',
        'raw_column': 'description',
        'decision': 'clean heavily',
        'why': 'Strong theme signal, but includes blanks, very short text, long outliers, and scraped spacing artifacts.',
        'planned_use': 'Cleaned keyword extraction and theme tagging; avoid direct raw text modeling without preprocessing.',
    },
    {
        'field': 'numRatings',
        'raw_column': 'numRatings',
        'decision': 'use cautiously',
        'why': 'Complete and numeric, but measures Goodreads visibility and age/platform bias, not total readership.',
        'planned_use': 'Visibility component of popularity score using log transform.',
    },
    {
        'field': 'averageRating',
        'raw_column': key_field_map['averageRating'],
        'decision': 'use cautiously',
        'why': 'Complete and bounded 0-5, but ratings are inflated/skewed by Goodreads community behavior.',
        'planned_use': 'Affection component of popularity score after numeric coercion.',
    },
    {
        'field': 'bbeVotes',
        'raw_column': 'bbeVotes',
        'decision': 'use cautiously',
        'why': 'Useful canonization signal, but has suspicious negative values and reflects a specific Goodreads list mechanism.',
        'planned_use': 'Canonization component of popularity score after coercion and invalid-value cleanup.',
    },
    {
        'field': 'bbeScore',
        'raw_column': 'bbeScore',
        'decision': 'ignore',
        'why': 'Complete and numeric but opaque; overlaps with BBE list ranking and is not part of the popularity formula.',
        'planned_use': 'Keep for reference/ranking sanity checks, but exclude from core model features.',
    },
])

feature_decisions


,field,raw_column,decision,why,planned_use
0,title,title,clean heavily,"Complete and valuable, but needs whitespace/ca...","Display label, duplicate checks, keyword/title..."
1,author,author,clean heavily,"Complete, but many rows contain multiple contr...",Extract primary author and optionally preserve...
2,firstPublishDate,firstPublishDate,clean heavily,"Essential for era grouping, but has many blank...",Create cleaned first_publish_year; filter to 1...
3,genres,genres,use cautiously,"Parseable list-like strings, but Goodreads she...","Input to custom theme buckets, combined with t..."
4,description,description,clean heavily,"Strong theme signal, but includes blanks, very...",Cleaned keyword extraction and theme tagging; ...
5,numRatings,numRatings,use cautiously,"Complete and numeric, but measures Goodreads v...",Visibility component of popularity score using...
6,averageRating,rating,use cautiously,"Complete and bounded 0-5, but ratings are infl...",Affection component of popularity score after ...
7,bbeVotes,bbeVotes,use cautiously,"Useful canonization signal, but has suspicious...",Canonization component of popularity score aft...
8,bbeScore,bbeScore,ignore,Complete and numeric but opaque; overlaps with...,"Keep for reference/ranking sanity checks, but ..."


## Initial Takeaways

- Treat empty strings as missing values.
- Treat ISBN `9999999999999` as missing.
- Preserve raw date strings and create cleaned date/year columns with explicit two-digit-year handling.
- Deduplicate with care: `bookId` duplicates are likely repeated records, while title duplicates can be valid distinct books or editions.
- Normalize list-like columns such as `genres` before analysis.
- Clean description spacing and flag blank, very short, and very long descriptions.
